Pull camera-violation tickets (codes 6, 37) for each parking-violations fiscal year dataset, using `fetch_soda` from `data/download.py` so every pull is cached in the local DuckDB and logged in `_provenance`.

Each fiscal year is fetched in its own cell. Datasets are tens of millions of rows before filtering, so keeping the pulls separate means a failed or interrupted run only has to redo the year it stopped on — `fetch_soda` serves any year already in the cache from local Parquet instead of hitting the API again. Running all of these cells the first time might take a while (~60-90 minutes), but then a local copy will be cached moving forward.

Note: if you use a different WHERE clause/change the parameters in fetch_soda(), new data will be pulled from the API.

see socrata API documentation here: https://dev.socrata.com/foundry/data.cityofnewyork.us/pvqr-7yc4

In [2]:
import sys
sys.path.append("../..")

from data.download import fetch_soda

WHERE = "violation_code IN (7, 36)"

In [2]:
fetch_soda("2bnn-yakx", "tickets_fy2017", WHERE)

(1917003, 38)

In [3]:
fetch_soda("a5td-mswe", "tickets_fy2018", WHERE)

(1631425, 39)

In [4]:
fetch_soda("faiq-9dfq", "tickets_fy2019", WHERE)

(1570054, 39)

In [5]:
fetch_soda("p7t3-5i9s", "tickets_fy2020", WHERE)

(4173023, 38)

In [6]:
fetch_soda("kvfd-bves", "tickets_fy2021", WHERE)

tickets_fy2021 rows: 4643494 rows [12:51, 6018.43 rows/s] 


(4643494, 38)

In [7]:
fetch_soda("7mxj-7a6y", "tickets_fy2022", WHERE)

tickets_fy2022 rows: 5298606 rows [20:02, 4407.59 rows/s]


(5298606, 39)

In [8]:
fetch_soda("869v-vr48", "tickets_fy2023", WHERE)

tickets_fy2023 rows: 8931848 rows [35:14, 4223.56 rows/s] 


(8931848, 38)

In [9]:
fetch_soda("8zf9-spf8", "tickets_fy2024", WHERE)

tickets_fy2024 rows: 6285732 rows [22:28, 4661.88 rows/s]


(6285732, 39)

In [10]:
fetch_soda("m5vz-tzqv", "tickets_fy2025", WHERE)

tickets_fy2025 rows: 5590777 rows [26:32, 3511.08 rows/s]


(5590777, 38)

In [11]:
fetch_soda("9mwx-gamw", "tickets_fy2026", WHERE)

tickets_fy2026 rows: 4834120 rows [20:46, 3878.49 rows/s]


(4834120, 40)

In [12]:
fetch_soda("pvqr-7yc4", "tickets_fy2027", WHERE)

tickets_fy2027 rows: 1004802 rows [04:13, 3962.96 rows/s]


(1004802, 29)

In [4]:
# combine each fiscal year as a view, so no second copy is stored (UNION ALL BY NAME handles the extra
# column some years have) instead of pd.concat, so ~20M rows never have to
# sit in pandas all at once
from data.base import get_connection

FISCAL_YEAR_TABLES = [
    "tickets_fy2017",
    "tickets_fy2018",
    "tickets_fy2019",
    "tickets_fy2020",
    "tickets_fy2021",
    "tickets_fy2022",
    "tickets_fy2023",
    "tickets_fy2024",
    "tickets_fy2025",
    "tickets_fy2026",
    "tickets_fy2027",
]

con = get_connection()
try:
    union_sql = " UNION ALL BY NAME ".join(
        f"SELECT * FROM {table}" for table in FISCAL_YEAR_TABLES
    )
    con.execute(f"CREATE OR REPLACE VIEW tickets_all AS {union_sql}")
    row_count = con.execute("SELECT COUNT(*) FROM tickets_all").fetchone()[0]
finally:
    con.close()

row_count

45880884